[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/research_assistant/langgraph.ipynb)

# Research assistant — LangGraph

**Task.** Answer which interventions reduce household food waste through a five-node, safety-screened research graph.

**Read this notebook as:** choose a provider → load evidence → declare graph nodes → plan, retrieve, synthesise and critique.

In [ ]:
# 1. Choose the model provider; then use Run All. No terminal command is needed.
MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "research-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = "models/local/Qwen3-0.6B-Q8_0.gguf"
API_MODEL_NAME = "gemini-3.1-flash-lite"
SAVE_API_CREDENTIAL = True  # saves hidden input to ignored .private/ storage
# Mock runtime is under one minute on CPU; local and API runs can be slower.

In [1]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.9", "pydantic==2.12.5"],
        check=True,
    )

In [2]:
import json
from pathlib import Path
from typing import ClassVar, Literal

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, ConfigDict, Field

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.notebook import prepare_gemini_api_key  # noqa: E402
from agentic_tutorial.safety import (  # noqa: E402
    PolicyOutcome,
    SafetyEngine,
    SafetyPolicy,
    UntrustedContent,
)
from agentic_tutorial.schemas import CriticDecision, Message, MessageRole  # noqa: E402

catalogue = json.loads((ROOT / "data/research_assistant/evidence_catalogue.json").read_text())
fixture_path = ROOT / "data/research_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
if MODEL_PROVIDER == "api":
    prepare_gemini_api_key(ROOT, save=SAVE_API_CREDENTIAL)

## Contracts and graph nodes

Model proposals cross typed boundaries. Retrieved passages remain untrusted; injection indicators remove a record before extraction. Each node returns explicit state, trace and usage.

In [3]:
# --- Declarations: typed outputs, model adapter, and workflow helpers ---
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class QueryPlan(Strict):
    schema_id: ClassVar[str] = "research.query.v1"
    queries: tuple[str, ...] = Field(min_length=1, max_length=4)


class Claim(Strict):
    source_id: str
    claim: str
    stance: Literal["supporting", "conflicting"]


class Ledger(Strict):
    schema_id: ClassVar[str] = "research.ledger.v1"
    claims: tuple[Claim, ...]


class Synthesis(Strict):
    schema_id: ClassVar[str] = "research.synthesis.v1"
    answer: str
    source_ids: tuple[str, ...]
    outcome: Literal["qualified_answer", "abstention"]


class CriticOutput(Strict):
    accepted: bool
    issues: tuple[str, ...] = ()


Claim.model_rebuild(_types_namespace={"Literal": Literal})
Ledger.model_rebuild(_types_namespace={"Claim": Claim})
Synthesis.model_rebuild(_types_namespace={"Literal": Literal})


def fresh_model():
    model_names = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}
    model_path = ROOT / LOCAL_MODEL_PATH if MODEL_PROVIDER == "local" else None
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, prompt):
    response = await client.generate([user(prompt)], response_schema=schema)
    return schema.model_validate(response.structured_output)


def search(query):
    terms = set(query.casefold().split())
    return [
        record
        for record in catalogue["records"]
        if terms & set((record["title"] + " " + record["passage"]).casefold().split())
    ]

In [4]:
def build_graph():
    client = fresh_model()
    safety = SafetyEngine(SafetyPolicy(allowed_tools=("catalogue_search",)))

    async def plan_node(state):
        plan = await propose(
            client,
            QueryPlan,
            "Plan searches for smaller plates, meal planning, and "
            "information-only campaigns; return at most four focused queries.",
        )
        return {
            "plan": plan,
            "trace": [{"event": "plan", "queries": plan.queries}],
            "model_calls": 1,
        }

    def retrieve_node(state):
        retrieved = {
            record["source_id"]: record
            for query in state["plan"].queries
            for record in search(query)
        }
        assessments = {
            sid: safety.inspect_retrieved_content(
                UntrustedContent(source_id=sid, text=record["passage"])
            )
            for sid, record in retrieved.items()
        }
        safe = {
            sid: record
            for sid, record in retrieved.items()
            if assessments[sid].decision.outcome in {PolicyOutcome.ALLOW, PolicyOutcome.TRANSFORM}
            and not assessments[sid].indicators
        }
        return {
            **state,
            "safe": safe,
            "trace": [
                *state["trace"],
                {
                    "event": "retrieve",
                    "source_ids": sorted(retrieved),
                    "isolated": [
                        sid for sid, assessment in assessments.items() if assessment.indicators
                    ],
                },
            ],
        }

    async def ledger_node(state):
        ledger = await propose(
            client,
            Ledger,
            f"Extract one source-grounded claim per record from {state['safe']}. "
            "Use supporting for reductions and conflicting for inconsistent effects.",
        )
        claims = tuple(claim for claim in ledger.claims if claim.source_id in state["safe"])
        return {
            **state,
            "claims": claims,
            "model_calls": 2,
            "trace": [
                *state["trace"],
                {
                    "event": "ledger",
                    "supporting": [c.source_id for c in claims if c.stance == "supporting"],
                    "conflicting": [c.source_id for c in claims if c.stance == "conflicting"],
                },
            ],
        }

    async def synthesis_node(state):
        answer = await propose(
            client,
            Synthesis,
            f"Synthesise only {state['claims']} with conditions, "
            "conflicts and their source IDs; use outcome qualified_answer.",
        )
        return {
            **state,
            "answer": answer,
            "model_calls": 3,
            "trace": [*state["trace"], {"event": "synthesis"}],
        }

    async def critic_node(state):
        critic = await propose(
            client,
            CriticOutput,
            f"Accept only if supported, qualified and cited: {state['answer'].model_dump()}",
        )
        critic = CriticDecision(accepted=critic.accepted, issues=critic.issues)
        citations_valid = set(state["answer"].source_ids).issubset(state["safe"])
        outcome = state["answer"].outcome if critic.accepted and citations_valid else "abstention"
        return {
            **state,
            "outcome": outcome,
            "model_calls": 4,
            "termination": "criteria_met",
            "trace": [
                *state["trace"],
                {
                    "event": "critic",
                    "accepted": critic.accepted,
                    "citations_valid": citations_valid,
                },
            ],
        }

    graph = StateGraph(dict)
    graph.add_node("plan", plan_node)
    graph.add_node("retrieve", retrieve_node)
    graph.add_node("ledger", ledger_node)
    graph.add_node("synthesise", synthesis_node)
    graph.add_node("critic", critic_node)
    graph.add_edge(START, "plan")
    graph.add_edge("plan", "retrieve")
    graph.add_edge("retrieve", "ledger")
    graph.add_conditional_edges(
        "ledger",
        lambda state: "synthesise" if state["claims"] else END,
        {"synthesise": "synthesise", END: END},
    )
    graph.add_edge("synthesise", "critic")
    graph.add_edge("critic", END)
    return graph.compile()


# --- Execution: run the workflow and evaluate its observable result ---
first = await build_graph().ainvoke({})
second = await build_graph().ainvoke({}) if MODEL_PROVIDER == "mock" else first
evaluation = {
    "component": len(first["claims"]) == 3,
    "trajectory": len(first["trace"]) == 5 and first["model_calls"] <= 4,
    "task": first["outcome"] == "qualified_answer",
    "safety": "fw-004" not in first["answer"].source_ids,
    "repeated_run": first == second,
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "outcome": first["outcome"],
    "resource_report": {"model_calls": first["model_calls"], "graph_nodes": 5},
    "fallback": "empty ledger routes directly to abstention",
}

{'evaluation': {'component': True,
  'trajectory': True,
  'task': True,
  'safety': True,
  'repeated_run': True},
 'trace': [{'event': 'plan',
   'queries': ('smaller plates', 'meal planning', 'information campaign')},
  {'event': 'retrieve',
   'source_ids': ['fw-001', 'fw-002', 'fw-003'],
   'isolated': []},
  {'event': 'ledger',
   'supporting': ['fw-001', 'fw-002'],
   'conflicting': ['fw-003']},
  {'event': 'synthesis'},
  {'event': 'critic', 'accepted': True, 'citations_valid': True}],
 'outcome': 'qualified_answer',
 'resource_report': {'model_calls': 4, 'graph_nodes': 5},
 'fallback': 'empty ledger routes directly to abstention'}

## Limitation
Graph topology makes control flow inspectable but does not guarantee semantic claim validity; the bounded catalogue and conservative validators remain essential.